In [0]:
# run_pipeline - exécute la chaîne Databricks dans l'ordre de sql/README.md.
# Hors périmètre : étape 0 (préparation locale) et étape 2 (envoi dans le volume), faites sur le PC.
import os 
import time

RACINE = os.getcwd()   # dossier de ce notebook = racine du dépôt
if not os.path.exists(os.path.join(RACINE, "sql", "README.md")):
    raise RuntimeError(f"Ce notebook doit être à la racine du dépôt (dossier actuel : {RACINE})")

# Couches à exécuter. "gold" reste désactivé tant que le contrat Gold (synchro 1) n'est pas validé.
COUCHES_ACTIVES = ["setup", "bronze", "silver", "gold"]
# Ordre de sql/README.md
ETAPES = [
    ("setup",   "sql/00_setup/01_catalogue.sql"),
    ("setup",   "sql/00_setup/02_droits_equipe.sql"),
    ("bronze",  "sql/dml/bronze/patient/01_patient.py"),
    ("bronze",  "sql/dml/bronze/diagnostic/01_diagnostic.py"),
    ("bronze",  "sql/dml/bronze/hospitalisation/01_hospitalisation.sql"),
    ("bronze",  "sql/dml/bronze/consultation/01_bronze_consultation.sql"),
    ("bronze",  "sql/dml/bronze/satisfaction/00.bronze_satisfaction.py"),
    ("bronze",  "sql/dml/bronze/professionnel_de_sante/02_bronze_professionnel_de_sante.sql"),
    ("bronze",  "sql/dml/bronze/specialite/03_bronze_specialites.sql"),
    ("bronze",  "sql/dml/bronze/etablissement/01_etablissement_sante.sql"),  
    ("bronze",  "sql/dml/bronze/etablissement/02_activite_professionnel_sante.sql"),
    ("bronze",  "sql/dml/bronze/referentiels/01_cog_departement.sql"),
    ("bronze",  "sql/dml/bronze/referentiels/02_cog_region.sql"), 
    ("bronze",  "sql/dml/bronze/deces/01_deces.sql"), 
    ("silver",  "sql/dml/silver/00_rejets.sql"),
    ("silver",  "sql/dml/silver/referentiels/01_ref_geographie.sql"),
    ("silver",  "sql/dml/silver/etablissement/01_etablissement.sql"),
    ("silver",  "sql/dml/silver/patient/01_patient.sql"),
    ("silver",  "sql/dml/silver/diagnostic/01_diagnostic.sql"),
    ("silver",  "sql/dml/silver/hospitalisation/01_hospitalisation.sql"),
    ("silver",  "sql/dml/silver/consultations/01_silver_specialite.sql"),
    ("silver",  "sql/dml/silver/consultations/02_silver_professionnel.sql"),
    ("silver",  "sql/dml/silver/consultations/03_silver_consultation.sql"),
    ("silver",  "sql/dml/silver/etablissement/02_pont_professionnel_etablissement.sql"),
    ("silver",  "sql/dml/silver/deces/01_deces.sql"),
    ("silver",  "sql/dml/silver/satisfaction/01_ref_region_esatis.sql"),
    ("silver",  "sql/dml/silver/satisfaction/02_satisfaction_esatis.sql"),
    ("silver",  "sql/dml/silver/satisfaction/03_ref_seuils_esatis.sql"),
    ("gold",    "sql/ddl/gold/01_dim_temps.sql"),
    ("gold",    "sql/ddl/gold/02_dim_patient.sql"),
    ("gold",    "sql/ddl/gold/03_dim_diagnostic.sql"),
    ("gold",    "sql/dml/gold/dim/dim_temps/01_dim_temps.sql"),
    ("gold",    "sql/dml/gold/dim/dim_patient/01_dim_patient.sql"),
    ("gold",    "sql/dml/gold/dim/dim_diagnostic/01_dim_diagnostic.sql"),
    ("qualite", "sql/quality/controles_qualite.sql"),
    ("qualite", "sql/dml/silver/consultations/10_controles_silver.sql"),

]


def decouper_sql(texte: str) -> list[str]:
    """Découpe un script en instructions sur les « ; »,
    sauf ceux placés entre apostrophes ou dans un commentaire."""
    instructions, courante = [], []
    i, dans_chaine = 0, False
    while i < len(texte):
        c = texte[i]
        if dans_chaine:
            courante.append(c)
            if c == "'":
                dans_chaine = False          # une apostrophe doublée '' referme puis rouvre aussitôt
        elif texte.startswith("--", i):      # commentaire de ligne : ignoré jusqu'à la fin de la ligne
            fin = texte.find("\n", i)
            i = len(texte) if fin == -1 else fin
            continue
        elif texte.startswith("/*", i):      # commentaire de bloc
            fin = texte.find("*/", i + 2)
            i = len(texte) if fin == -1 else fin + 2
            continue
        elif c == "'":
            dans_chaine = True
            courante.append(c)
        elif c == ";":
            instructions.append("".join(courante).strip())
            courante = []
        else:
            courante.append(c)
        i += 1
    instructions.append("".join(courante).strip())
    return [s for s in instructions if s]


def executer_sql(chemin: str, afficher: bool) -> None:
    with open(os.path.join(RACINE, chemin), encoding="utf-8") as f:
        for instruction in decouper_sql(f.read()):
            resultat = spark.sql(instruction)
            if afficher and instruction.upper().startswith(("SELECT", "WITH", "SHOW")):
                display(resultat)


def executer_python(chemin: str) -> None:
    complet = os.path.join(RACINE, chemin)
    if os.path.exists(complet):                  # simple fichier Python
        with open(complet, encoding="utf-8") as f:
            code = f.read()
        exec(compile(code, chemin, "exec"),
             {"spark": spark, "dbutils": dbutils, "display": display, "__name__": "__main__"})
    else:                                        # notebook : Databricks le stocke sans l'extension .py
        dbutils.notebook.run(f"./{chemin[:-3]}", 3600)


bilan = []
for couche, chemin in ETAPES:
    if couche not in COUCHES_ACTIVES:
        bilan.append((couche, chemin, "IGNORÉE", 0.0))
        continue
    print(f"▶ {couche:<8} {chemin}")
    debut = time.time()
    try:
        if chemin.endswith(".sql"):
            executer_sql(chemin, afficher=(couche == "qualite"))
        else:
            executer_python(chemin)
        bilan.append((couche, chemin, "OK", time.time() - debut))
    except Exception as e:
        bilan.append((couche, chemin, "ÉCHEC", time.time() - debut))
        print(f"✖ {type(e).__name__} : {str(e)[:500]}")
        break

print("\nRécapitulatif")
for couche, chemin, statut, duree in bilan:
    print(f"  {statut:<8} {couche:<8} {chemin:<66} {duree:7.1f} s")

if bilan and bilan[-1][2] == "ÉCHEC":
    raise RuntimeError("Chaîne interrompue : voir l'étape en ÉCHEC ci-dessus")

In [0]:
%sql
SELECT 'dim_patient' AS t, count(*) AS lignes, count(DISTINCT patient_sk) AS cles_uniques,
       count_if(patient_sk = -1) AS inconnu
FROM chu.gold.dim_patient
UNION ALL
SELECT 'dim_diagnostic', count(*), count(DISTINCT diagnostic_sk), count_if(diagnostic_sk = -1)
FROM chu.gold.dim_diagnostic;

In [0]:
%sql
SELECT max(int(split(valeur, '/')[0])) AS max_bloc_1,
       max(int(split(valeur, '/')[1])) AS max_bloc_2
FROM chu.silver.rejets
WHERE table_cible = 'hospitalisation';

In [0]:
%sql
SELECT table_schema, table_name, table_owner, last_altered
FROM chu.information_schema.tables
WHERE table_schema IN ('silver', 'gold', 'quality', 'bench')
ORDER BY table_schema, table_name;